# Notebook 1: Identify Crop And Part With Router

Bu notebook sadece router yolunu calistirir: bootstrap, model erisim kontrolu, opsiyonel on-yukleme ve tek goruntu analizi.

Notebook 1 hastalikli goruntulerde dogru **bitki/crop yonlendirmesi** icin tasarlanmistir; hastalik sinifi tahmini yapmaz.

Router ayni runtime icinde bir kez hazirlanir ve sonraki denemelerde yeniden kullanilir. Boylece her yeni goruntu denemesinde modeller tekrar indirilmez veya yeniden yuklenmez.

Donen sonuc ust seviyede `crop` ve `part` alanlarini korur. Ayrica `router` blogu icinde router durumu, birincil tespit ve tespit sayisi bulunur. Bu notebook crop adapter yuklemez.

Bu surumde varsayilan router profili `balanced` olacak sekilde ayarlanmistir ve tanisal cikti (aday crop'lar, margin, unknown/rejection sinyalleri) gorunur hale getirilmistir. Bu tercih, acik-kume ve secmeli-reddetme literaturunde belirsiz durumda asiri guvenli etiket yerine daha kontrollu karar verilmesi geregiyle uyumludur (Hendrycks & Gimpel, 2017; Guo et al., 2017; Geifman & El-Yaniv, 2019).


## Hizli Akis

Bu notebook sadece router ile crop/part bulur; hastalik sinifi veya adapter tahmini yapmaz.

- Yeni kullanici icin `NOTEBOOK_PROFILE = 'a100_colab_default'` ve `IMAGE_PATH = None` yeterlidir; analiz hucresi upload ister.
- Ayni runtime icinde baska goruntu denemek icin sadece analiz hucresini tekrar calistirin.
- `unknown` veya `router_uncertain` sonucu bir hata degil; routerin guvenli reddetme durumudur.
- Adapterli hastalik tahmini icin Notebook 3 veya Notebook 4 kullanin.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')


def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        marker = candidate / 'scripts' / 'notebook_helpers' / 'cell_script_runner.py'
        if marker.is_file():
            repo_root = candidate.resolve()
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    return CLONE_TARGET


_ensure_aads_repo_on_path()

from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb1_cell01_bootstrap.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script
run_cell_script('nb1_cell02_access_check.py', globals())


In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script

# Kullanici etkileşimli ayarlar
NOTEBOOK_PROFILE = 'a100_colab_default'  # custom, a100_colab_default, cpu_debug
NOTEBOOK_SPEED_MODE = 'fast'  # fast, balanced, quality

# Gerekirse son override katmani (opsiyonel)
INFERENCE_OVERRIDES = {}

run_cell_script('nb1_cell03_runtime_setup.py', globals())

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script

# Kullanici etkileşimli ayarlar
ANALYSIS_IMAGE_PATH = IMAGE_PATH  # Yeni dosya yolu verin veya None birakip yukleme kullanin.

run_cell_script('nb1_cell04_analysis.py', globals())

result

In [ ]:
from scripts.notebook_helpers.cell_script_runner import run_cell_script

# Kullanici etkileşimli ayarlar
ROUTER_EVAL_ROOT = None  # Ornek: '/content/router_part_eval'
ROUTER_EVAL_OUTPUT = None  # Ornek: '/content/router_part_eval_summary.json'
RUN_ROUTER_EVAL = False
EVAL_MIN_CONFIDENCE_GRID = '0.20,0.25,0.30,0.35,0.40,0.45,0.50'
EVAL_MARGIN_GRID = '0.00,0.02,0.05,0.08,0.10,0.12,0.15'

run_cell_script('nb1_cell05_router_eval.py', globals())